# 06 - Pellet scoring validation (manual vs algorithmic)

**Purpose**: measure how well the algorithmic pellet-outcome classifier
agrees with manual scoring, as a prerequisite for trusting the automated
outcomes in the main analysis.

**Restricted to pillar trays** because the algorithm is designed for
that tray type only.

**Input**: ``data.manual_pelletdf`` and ``data.kinematicsdf``.


In [ ]:
# parameters
FIGSIZE_CM = (14, 5)
FIGSIZE_PER_PHASE = (9, 5)


In [ ]:
import pandas as pd                                              # dataframes
import matplotlib.pyplot as plt                                  # plotting
import seaborn as sns                                            # heatmap helper used for the confusion matrix
from sklearn.metrics import cohen_kappa_score, confusion_matrix  # chance-corrected agreement statistic + raw count matrix

from endpoint_ck_analysis.config import EXAMPLE_OUTPUT_DIR       # PNG output dir
from endpoint_ck_analysis.data_loader import load_all            # one-shot loader
from endpoint_ck_analysis.helpers.plotting import stamp_version  # figure version footer

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)            # ensure output folder exists
data = load_all()                                                # uses cache from 00_setup


## 1. Build the per-pellet validation dataframe

Inner-join manual and algorithmic scores on shared per-pellet keys.


In [ ]:
manual_pellet_pillar = data.manual_pelletdf[data.manual_pelletdf['tray_type'] == 'P']           # restrict to pillar trays only -- the algorithm wasn't designed for E/F trays
kinematics_pillar = data.kinematicsdf[data.kinematicsdf['tray_type'] == 'P']                    # same restriction on the kinematics side

algo_per_segment = kinematics_pillar[                                                            # pull just the columns we need to identify segments + their algorithmic outcome...
    ['subject_id', 'session_date', 'tray_type', 'run_number', 'segment_num', 'segment_outcome']
].drop_duplicates().rename(columns={'run_number': 'tray_number', 'segment_num': 'pellet_number'}) # ...drop_duplicates collapses multiple reaches in the same segment to one row, then rename to the column names manual scoring uses

validation = manual_pellet_pillar[                                                              # build the validation join key from manual scoring's columns...
    ['subject_id', 'session_date', 'tray_type', 'tray_number', 'pellet_number', 'score']
].merge(                                                                                         # ...and inner-join against the algorithmic per-segment table
    algo_per_segment,
    on=['subject_id', 'session_date', 'tray_type', 'tray_number', 'pellet_number'],             # composite key uniquely identifies one pellet event
    how='inner',                                                                                 # 'inner' drops pellets that don't match in both sources; that's intentional -- only paired observations count for agreement
)
print(f'Matched {len(validation)} pillar pellets between manual and algorithmic scoring')

manual_cat_map = {0: 'missed', 1: 'displaced', 2: 'retrieved'}                                  # manual scoring uses integer codes; map to readable strings for the confusion matrix
algo_cat_map = {                                                                                 # algorithmic scoring has more granular categories; collapse to the same three buckets manual uses
    'untouched': 'missed', 'uncertain': 'missed',
    'displaced_sa': 'displaced', 'displaced_outside': 'displaced',
    'retrieved': 'retrieved',
}
validation['manual_cat'] = validation['score'].map(manual_cat_map)                              # .map applies the dict element-wise; result is a new Series of category strings
validation['algo_cat'] = validation['segment_outcome'].map(algo_cat_map)
validation['manual_contacted'] = validation['manual_cat'] != 'missed'                           # boolean: did the manual score say the pellet was at least touched?
validation['algo_contacted'] = validation['algo_cat'] != 'missed'                               # same boolean for the algorithm; binary metric is easier to interpret than three-way


## 2. Summary statistics


In [ ]:
three_way = (validation['manual_cat'] == validation['algo_cat']).mean()                           # element-wise equality on the three categories -> boolean Series; .mean() of bools = fraction True = exact agreement rate
binary = (validation['manual_contacted'] == validation['algo_contacted']).mean()                  # same logic on the binary contacted/missed collapse
kappa = cohen_kappa_score(validation['manual_cat'], validation['algo_cat'])                       # Cohen's kappa: agreement adjusted for the rate you'd expect by chance given each rater's marginal distribution
print(f'Three-way exact agreement:     {three_way:.3%}')                                          # .3% formats as percentage with 3 decimal places
print(f'Binary (contacted vs missed):  {binary:.3%}')
print(f"Cohen's kappa (three-way):     {kappa:.3f}")
print('Interpretation of kappa: <0.4 poor, 0.4-0.6 moderate, 0.6-0.8 substantial, >0.8 almost perfect') # standard kappa-interpretation thresholds (Landis & Koch 1977)
print('\nConfusion matrix (rows=manual, cols=algorithmic):')
print(pd.crosstab(validation['manual_cat'], validation['algo_cat'], margins=True))                # pandas crosstab counts co-occurrences; margins=True adds row/column totals


**What you just saw.** Three numbers and a count table.
- **Three-way exact agreement:** raw rate of "manual and algorithm
  gave the same one of {missed, displaced, retrieved}".
- **Binary agreement:** rate of "manual and algorithm agree on
  whether the pellet was touched at all" (collapsing displaced
  and retrieved into one bucket).
- **Cohen's kappa:** chance-corrected agreement on the three-way
  classification. The legend below the print spells out the
  standard ranges.

**What different patterns mean.**
- Kappa > 0.8 = the algorithmic outcome labels are trustworthy
  for publication; downstream analyses that use them are on
  solid ground.
- Kappa 0.6-0.8 = substantial agreement; algorithmic labels
  usable for most analyses, spot-check edge cases that hinge on
  retrieval-vs-displacement distinctions.
- Kappa 0.4-0.6 = moderate; flag any conclusion that depends on
  three-way categorization.
- Kappa < 0.4 = poor; default to manual scoring for the paper.
- Binary agreement much higher than three-way = algorithm is
  good at detecting contact but bad at distinguishing displaced
  vs retrieved. **What this means for the paper:** analyses
  using `contact_group` are fine; analyses using `outcome_group`
  (the three-way split) need extra caution.

**Reading the confusion matrix below.** The crosstab shows where
the disagreements live. Off-diagonal mass concentrated in the
displaced<->retrieved cells = the boundary the algorithm is
struggling with. Off-diagonal mass in missed<->contacted cells
= the algorithm is failing at contact detection itself, which
is more concerning.


## 3. Confusion matrix heatmap (counts + row-normalized)


In [ ]:
cats = ['missed', 'displaced', 'retrieved']                                                     # explicit category order; passing labels= to confusion_matrix locks the row/col axis to this order regardless of which categories appear in the data
cm = confusion_matrix(validation['manual_cat'], validation['algo_cat'], labels=cats)             # 3x3 raw-count matrix; rows=manual labels, cols=algorithmic labels
cm_norm = cm / cm.sum(axis=1, keepdims=True)                                                     # row-normalize: divide each row by its sum -> rows now sum to 1.0; reads as "given the manual class, what fraction does the algorithm assign to each class"; keepdims=True keeps it as a (3,1) for proper broadcasting

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_CM)                                              # two side-by-side panels: counts and normalized
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cats, yticklabels=cats,          # annot=True writes the count in each cell; fmt='d' formats as integer
            ax=axes[0], cbar_kws={'label': 'count'})
axes[0].set_xlabel('Algorithmic classification')
axes[0].set_ylabel('Manual classification')
axes[0].set_title('Pillar confusion matrix (raw counts)')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=cats, yticklabels=cats,   # fmt='.2%' formats as percentage with 2 decimals; vmin/vmax fix the colormap range so two heatmaps with different absolute scales remain visually comparable
            ax=axes[1], vmin=0, vmax=1, cbar_kws={'label': 'fraction'})
axes[1].set_xlabel('Algorithmic classification')
axes[1].set_ylabel('Manual classification')
axes[1].set_title('Pillar confusion matrix (row-normalized)')
plt.tight_layout()
stamp_version(fig, label='06 confusion')
plt.savefig(EXAMPLE_OUTPUT_DIR / '06_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


**What this figure shows.** Two heatmaps, both with manual class
on the y-axis and algorithmic class on the x-axis. The left panel
shows raw counts; the right panel shows the same matrix with
each row normalized to 1.0 (so the row reads as "given the
manual class, what fraction did the algorithm assign to each
bucket").

**What different patterns mean.**
- Strong diagonal in the row-normalized panel (each diagonal
  cell close to 100%) = the algorithm is rarely wrong about
  any of the three categories.
- Off-diagonal mass concentrated in displaced<->retrieved cells
  = the boundary between successful retrieval and "knocked the
  pellet but didn't bring it in" is what's hard. *Example:* if
  the manual=retrieved row shows 70% retrieved, 25% displaced,
  5% missed in the algo classification, the algorithm is
  underestimating retrievals by treating them as displacements.
- Off-diagonal mass in missed<->contacted cells = contact
  detection itself is failing. More concerning because contact
  is the precondition for any kinematic analysis.
- The raw-counts panel reveals class imbalance: if "missed" has
  tiny bin counts compared to the others, the kappa for that
  row is unstable even if the percentages look fine.

**Scientific reading.** Use the row-normalized panel to identify
which boundary the algorithm needs to be retrained on. Use the
raw-counts panel to know which classes have enough data to
trust the agreement statistic.


## 4. Per-phase agreement

Does the algorithm's accuracy drift across the experimental phases?


In [ ]:
validation_with_phase = validation.merge(                                                       # left-join phase_group onto each validated pellet so we can group by phase
    manual_pellet_pillar[['subject_id', 'session_date', 'tray_number', 'pellet_number', 'phase_group']],
    on=['subject_id', 'session_date', 'tray_number', 'pellet_number'],                          # composite key matches the validation table
    how='left',                                                                                  # keep every validation row even if phase isn't found (rare; would show as NaN phase_group)
)
per_phase = validation_with_phase.groupby('phase_group').apply(                                 # one row per phase
    lambda g: pd.Series({                                                                        # lambda runs on each phase's subset 'g'; returns a small Series of statistics
        'n': len(g),                                                                             # how many pellets contributed to this phase's number
        'three_way_agreement': (g['manual_cat'] == g['algo_cat']).mean(),                       # exact-agreement rate within phase
        'binary_agreement': (g['manual_contacted'] == g['algo_contacted']).mean(),              # contacted-vs-missed agreement within phase
    })
).sort_values('n', ascending=False)                                                              # sort by sample size so phases with the most data appear first
print(per_phase)

fig, ax = plt.subplots(figsize=FIGSIZE_PER_PHASE)                                               # one panel
x = range(len(per_phase))                                                                        # integer x positions (one per phase)
ax.bar([i - 0.2 for i in x], per_phase['three_way_agreement'], width=0.4,                        # offset by -0.2 so this bar sits to the LEFT of the phase tick
       label='Three-way agreement', color='steelblue')
ax.bar([i + 0.2 for i in x], per_phase['binary_agreement'], width=0.4,                           # offset by +0.2 so this bar sits to the RIGHT, paired with the matching three-way bar
       label='Binary agreement', color='orange')
ax.set_xticks(list(x))
ax.set_xticklabels(per_phase.index, rotation=45, ha='right')                                    # rotate phase names so they don't overlap; ha='right' anchors text at the right edge
ax.set_ylabel('Agreement rate')
ax.set_ylim(0, 1)                                                                                # agreement is a fraction; lock axis to [0,1] so the y-axis is comparable across runs
ax.axhline(0.9, color='green', linestyle='--', linewidth=0.7, label='0.90 reference')           # green dashed line at 0.9 -- a typical "good agreement" benchmark
ax.legend()
ax.set_title('Manual vs algorithmic agreement by phase (pillar trays only)')
for i, (phase, row) in enumerate(per_phase.iterrows()):                                          # annotate each phase position with its N at the bottom of the plot
    ax.text(i, 0.02, f"N={int(row['n'])}", ha='center', fontsize=8)
plt.tight_layout()
stamp_version(fig, label='06 per phase')
plt.savefig(EXAMPLE_OUTPUT_DIR / '06_agreement_by_phase.png', dpi=150, bbox_inches='tight')
plt.show()


**What this figure + printout show.** The print is per-phase
agreement (n pellets, three-way rate, binary rate). The figure is
the same data as side-by-side bars: blue = three-way, orange =
binary. The green dashed line at 0.9 is a "good agreement"
reference.

**What different patterns mean.**
- Steady high agreement across all phases (both colors near or
  above the green line) = the algorithm is robust across
  experimental phases. Use it freely throughout the analysis.
- Agreement dropping post-injury = the injured cohort's reach
  kinematics differ from what the algorithm was trained on, so
  its outcome decisions are less reliable on those phases.
  *Example:* if Baseline kappa is 0.9 but Post_Injury_1 binary
  agreement falls to 0.6, post-injury outcome labels are the
  weakest part of the dataset; any kinematic finding that
  conditions on outcome at that phase needs a caveat.
- Three-way bar consistently lower than the binary bar at every
  phase = the algorithm reliably detects contact but consistently
  struggles with the displaced/retrieved boundary. Stays
  phase-independent; not a phase-specific issue.
- Phases with very small n (printed in the table) = agreement
  estimate at that phase is itself unreliable; ignore the bar
  and focus on phases with substantive sample sizes.

**Scientific reading.** This figure tells the paper which phases
you can trust the algorithm on. Where it dips, plan to either
manually re-score that phase's data or footnote the
phase-specific uncertainty in the discussion.
